[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/AD7031/blob/main/notebooks/week03_markowitz.ipynb)

In [ ]:
%pip install -q gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB

params = {
    "WLSACCESSID": "여기에 ACCESSID 붙여넣기",
    "WLSSECRET":   "여기에 SECRET 붙여넣기",
    "LICENSEID":   123456,   # 숫자 그대로, 따옴표 없이
}
env = gp.Env(params=params)

# Markowitz portfolio — a first model under uncertainty

In the week 2 LP every number was known. Now the data are uncertain: split a budget across 3 assets whose returns we will only learn **tomorrow**. All we have today are their **means** and **covariances**:

| asset | expected return $\mu_j$ | variance $\Sigma_{jj}$ |
| :--- | ---: | ---: |
| bond   | 6%  | 0.010 |
| tech   | 10% | 0.040 |
| energy | 15% | 0.090 |

Covariances: $\Sigma_{\text{bond,tech}} = 0.001$, $\;\Sigma_{\text{tech,energy}} = 0.015$, $\;\Sigma_{\text{bond,energy}} = 0$.

$x_j$ = share of the budget in asset $j$.

$$
\begin{aligned}
\min_{x} \quad & x^\top \Sigma\, x \\
\text{s.t.} \quad & \mu^\top x \ge 0.10 \\
                  & x_{\text{bond}} + x_{\text{tech}} + x_{\text{energy}} = 1 \\
                  & x_j \ge 0
\end{aligned}
$$

Minimize **risk** — the variance of the portfolio return — while the **expected** return reaches the 10% target.

Two firsts compared to the LP: the objective is **quadratic**, not linear — and it is still **convex**, so Gurobi solves it just as reliably. Written out term by term:

$$
x^\top \Sigma\, x \;=\; 0.010\,x_{\text{bond}}^2 + 0.040\,x_{\text{tech}}^2 + 0.090\,x_{\text{energy}}^2
  \;+\; 2(0.001)\,x_{\text{bond}}x_{\text{tech}} \;+\; 2(0.015)\,x_{\text{tech}}x_{\text{energy}}
$$

## Live coding — fill in the blanks

Skeleton for building the model live in class, one comment at a time.

In [ ]:
pf = gp.Model("Markowitz")

# decision variables, three continuous weights --- e.g., pf.addVar(vtype=GRB.CONTINUOUS)


# objective: portfolio variance --- quadratic! products of variables are allowed --- pf.setObjective()


# constraints: expected return >= 10%, weights sum to 1 --- pf.addConstr()


# model is written down -- let's read it back before solving --- pf.update(), pf.write("markowitz.lp"), print(open("markowitz.lp").read())



In [ ]:
# Now that we are done with writing the optimization model on the computer, let's solve the problem --- pf.optimize()


# Let's print the optimal weights and the risk



## Reference: the completed model

By hand — no arrays, no loops, no `quicksum` — so every term matches the math above.

In [ ]:
pf = gp.Model("markowitz")

# decision variables -- one weight per asset, continuous, lower bound 0 by default
x_bond   = pf.addVar(vtype=GRB.CONTINUOUS, name="x_bond")
x_tech   = pf.addVar(vtype=GRB.CONTINUOUS, name="x_tech")
x_energy = pf.addVar(vtype=GRB.CONTINUOUS, name="x_energy")

# objective -- portfolio variance, written out term by term
pf.setObjective(
    0.010 * x_bond * x_bond
    + 0.040 * x_tech * x_tech
    + 0.090 * x_energy * x_energy
    + 2 * 0.001 * x_bond * x_tech
    + 2 * 0.015 * x_tech * x_energy,
    GRB.MINIMIZE,
)

# constraints -- expected return meets the 10% target; weights sum to 1
pf.addConstr(0.06 * x_bond + 0.10 * x_tech + 0.15 * x_energy >= 0.10, name="target_return")
pf.addConstr(x_bond + x_tech + x_energy == 1, name="budget")

# read the model back before solving -- the quadratic objective shows up under [ ... ]/2
pf.update()
pf.write("markowitz.lp")
print(open("markowitz.lp").read())


In [ ]:
# Now that we are done with writing the optimization model on the computer, let's solve the problem
pf.optimize()

print(f"\nbond={x_bond.X:.2f}  tech={x_tech.X:.2f}  energy={x_energy.X:.2f}")
print(f"risk (variance) = {pf.ObjVal:.4f}")


## Plot the solution

One bar per asset, height $= x_j$ — the share of the budget it gets.

**The optimum is not at a corner.** The quadratic objective settles in the *interior* of a face — every asset gets a share. Compare the week 2 LP, where the answer sat on a vertex.

Later: sweep the target return from 6% to 15% and re-solve — the resulting (risk, return) curve is the **efficient frontier** (week 4).

In [ ]:
import matplotlib.pyplot as plt

assets  = ["bond", "tech", "energy"]
weights = [x_bond.X, x_tech.X, x_energy.X]
colors  = ["#4C72B0", "#DD8452", "#55A868"]

plt.figure(figsize=(6, 4))
plt.bar(assets, weights, color=colors)
plt.ylim(0, 1)
plt.ylabel("share of budget ($x_j$)")
plt.title(f"Optimal portfolio — risk (variance) = {pf.ObjVal:.4f}")
plt.tight_layout()
plt.show()

## Same model, less typing — arrays + `gp.quicksum`

Hand-typing does not scale: 50 assets means 50 variable lines and a 2,500-term objective.

Loop over arrays instead — same answer, harder to see which term is which. Commented out for now.

In [ ]:
# mu    = [0.06, 0.10, 0.15]        # 자산별 기대수익률
# Sigma = [[0.010, 0.001, 0.000],
#          [0.001, 0.040, 0.015],
#          [0.000, 0.015, 0.090]]   # 수익률 공분산행렬
# r = 0.10                          # 목표 기대수익률
# n = len(mu)
#
# pf2 = gp.Model("markowitz_arrays")
# x = pf2.addVars(n, lb=0.0, name="x")
# pf2.setObjective(gp.quicksum(Sigma[i][j] * x[i] * x[j] for i in range(n) for j in range(n)), GRB.MINIMIZE)
# pf2.addConstr(gp.quicksum(mu[i] * x[i] for i in range(n)) >= r)
# pf2.addConstr(x.sum() == 1)
# pf2.optimize()
# print([round(x[i].X, 2) for i in range(n)], round(pf2.ObjVal, 4))